## Task 1: Groq API Setup

In [1]:
# pip install -r requirements.txt
# Create .env:
# GROQ_API_KEY=your_groq_api_key_here

from langchain_groq import ChatGroq
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

## Load ChatGroq model
llm = ChatGroq(model='openai/gpt-oss-120b')

# Send prompt to Groq LLM
response = llm.invoke('Hello')

# Print model response
print(response.content)

Hello! How can I help you today?


## Task 2: Build a Groq Chatbot

In [2]:
from langchain_core.messages import SystemMessage, HumanMessage

# Load Groq model
llm = ChatGroq(model="openai/gpt-oss-120b")

# Create chatbot function
def groq_chat(prompt: str):
    human_msg = HumanMessage(content=prompt)
    system_msg = SystemMessage('You are a helpful AI assistant."')
    messages = [system_msg, human_msg]
    response = llm.invoke(messages)
    return response.content

# Test chatbot
print(groq_chat("What is Python?"))
print(groq_chat("What is Machine Learning?"))
print(groq_chat("What is RAG?"))

**Python** is a high‑level, interpreted programming language known for its readability and versatility. Created by Guido van Rossum and first released in 1991, Python emphasizes clear syntax (often described as “executable pseudocode”) and a philosophy of simplicity—summarized in the “Zen of Python” (`import this`).

### Key characteristics
| Feature | Why it matters |
|---------|----------------|
| **Readability** | Uses indentation to define code blocks, making programs easy to understand and maintain. |
| **Dynamic typing** | Variables don’t need explicit type declarations; the interpreter infers types at runtime. |
| **Rich standard library** | Batteries‑included modules for tasks like file I/O, networking, web services, data serialization, and more. |
| **Extensive ecosystem** | Thousands of third‑party packages (via PyPI) for web development (Django, Flask), data science (NumPy, pandas, scikit‑learn), AI/ML (TensorFlow, PyTorch), automation, scientific computing, etc. |
| **Cross

## Task 3: Groq-Based RAG Pipeline

In [8]:
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Create documents
documents = [
    Document(page_content="Python is a programming language."),
    Document(page_content="Machine Learning allows computers to learn from data."),
    Document(page_content="RAG retrieves relevant documents and gives them to an LLM."),
    Document(page_content="FAISS is used for similarity search on vector embeddings.")
]

# Loading embedding model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Store documents in FAISS
vector_store = FAISS.from_documents(documents, embeddings)

# Create retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# Load Groq model
llm = ChatGroq(model="openai/gpt-oss-120b")

# Create prompt
prompt = ChatPromptTemplate.from_template("""
Answer the question using the context below.

Context:
{context}

Question:
{question}
""")

parser = StrOutputParser()

# Format retrieved documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Create Retriever chain
retriever_chain = {'context': retriever | RunnableLambda(format_docs), 'question': RunnablePassthrough()}

# Create RAG chain
rag_chain = retriever_chain | prompt | llm | parser

# 9. Ask a question
answer = rag_chain.invoke("What is RAG?")
print(answer)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2948.63it/s]


RAG (Retrieval‑Augmented Generation) is a technique in which a system first retrieves relevant documents (or other external information) and then feeds those documents to a large language model (LLM) so that the model can use the retrieved content to produce more accurate, up‑to‑date, or knowledge‑rich responses.


## Task 4: Prompt Template for Groq RAG

In [9]:
# Create structured prompt
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a helpful assistant.
        Answer only using the provided context.
        If the answer is not available in the context,
        say: I don't know based on the provided context."""
    ),
    (
        "human",
        """Context:
        {context}

        Question:
        {question}"""
    )
])

# using same format docs as above task
retriever_chain = {'context': retriever | RunnableLambda(format_docs), 'question': RunnablePassthrough()}

# Create RAG chain
rag_chain = retriever_chain | prompt | llm | parser

# Test questions
print(rag_chain.invoke("What is Python?"))
print(rag_chain.invoke("What is FAISS?"))
print(rag_chain.invoke("What is the capital of Japan?"))

Python is a programming language.
FAISS is a tool used for similarity search on vector embeddings.
I don't know based on the provided context.


## Task 5: Create FastAPI Application

In [ ]:
## Please refer folder for main.py
## uvicorn main:app --reload

## Task 6: FastAPI + Groq Integration

In [ ]:
## Please refer folder for task6.py
## uvicorn task:app --reload

## Task 7: Run & Test FastAPI App Locally

In [ ]:
# Run the FastAPI application using:

# uvicorn task:app --reload

# Test the endpoints using the browser or Swagger UI.

# Swagger UI: http://127.0.0.1:8000/docs

# Endpoints:

# GET /health — Checks API status.

# POST /chat — Sends a question to the Groq LLM and returns the answer.

# Sample request:

# {
#   "query": "What is RAG?"
# }

# Verify that the API returns the expected response.

## Task 8: Production Readiness

In [ ]:
# Created requirements.txt containing the required Python packages.

# Stored the Groq API key in the .env file.

# Used python-dotenv to load environment variables.

# Added .env and .venv/ to .gitignore.

# Configured basic logging using Python's logging module.

# Added error handling using FastAPI's HTTPException.

# Added request validation using Pydantic models.

## Task 9: End-to-End Demo

In [ ]:
# Started the FastAPI application using Uvicorn.

# Tested the /health endpoint.

# Tested the /chat endpoint with different questions.

# Verified Groq-generated responses.

# Tested request validation for empty and oversized queries.

# Verified error handling for failed Groq API requests.

# Images are attached for reference

## Task 10: Observations & Insights

1. Why is Groq suitable for real-time applications?

Groq provides fast LLM inference, making it suitable for applications that require low-latency responses, such as chatbots, AI assistants, and real-time question-answering systems.

2. Groq vs OpenAI Latency Comparison (Conceptual)

Groq: Uses specialized inference hardware to generate responses quickly.
OpenAI: Provides hosted models with performance that varies depending on the model and serving infrastructure.

Actual latency depends on the model, prompt length, output length, network conditions, and service load.

3. Benefits of API-First GenAI Architecture

1. Modularity: The frontend and backend can be developed independently.
2. Reusability: The same AI service can be used by web and mobile applications.
3. Scalability: The backend can be scaled based on demand.
4. Maintainability: AI logic can be updated without changing the client application.
5. Integration: APIs make it easier to connect AI features with other services.
6. Security: API keys and model calls can remain on the backend rather than being exposed to clients.